# Day 33 — Evaluating an agent (golden cases)

"It seemed fine" is not a test. Pin behavior with **golden cases** and a **regression
gate** so a prompt tweak can't silently break things. We use a deterministic keyword
judge here so it runs offline; swap in `shared.evals.llm_judge` for open-ended answers.
Uses [`shared.evals`](../../shared/evals.py). ✅ Offline.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. The solution is below — try first.

In [ ]:
from shared.evals import GoldenCase, run_eval, regression_gate

def keyword_judge(question, answer, criteria):
    kws = [k.strip().lower() for k in criteria.split(",")]
    hit = sum(1 for k in kws if k in answer.lower())
    return hit / len(kws), f"{hit}/{len(kws)} keywords"

def agent_fn(q):
    table = {"capital of france": "Paris is the capital of France."}
    return table.get(q.lower(), "I don't know")

cases = [
    GoldenCase("france", "capital of France", "paris"),
    GoldenCase("unknown", "capital of Atlantis", "paris"),  # will score 0
]
# TODO: results = run_eval(agent_fn, cases, judge=keyword_judge)
#       then call regression_gate(results, threshold=0.8)


## 🔒 Solution

One correct way to do it.

In [ ]:
from shared.evals import GoldenCase, run_eval, regression_gate

def keyword_judge(question, answer, criteria):
    kws = [k.strip().lower() for k in criteria.split(",")]
    hit = sum(1 for k in kws if k in answer.lower())
    return hit / len(kws), f"{hit}/{len(kws)} keywords"

def agent_fn(q):
    table = {"capital of france": "Paris is the capital of France."}
    return table.get(q.lower(), "I don't know")

cases = [
    GoldenCase("france", "capital of France", "paris"),
    GoldenCase("unknown", "capital of Atlantis", "paris"),
]
results = run_eval(agent_fn, cases, judge=keyword_judge)
regression_gate(results, threshold=0.8)   # expect FAIL (one case scores 0)